<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-13-capstone-defend-documind/lesson-13.4-capstone-deploy/notebooks/GCP_Capstone_13.4_CapstoneDeploy.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13.4 — Build Phase 3: Your Own Pipeline, an SLO You Can Hold, and a Runbook

The eighth component is a recorded rollback with evidence. This phase stands up WIF on your project, gets one green release through the eval gate onto a candidate revision and into traffic, rolls it back with the same flip, and writes the runbook a colleague could follow at 3am while worried.


## Cell 1: Keyless, on Your Project

Five values, and the two that fail in opposite directions.


In [ ]:
# Keyless CI/CD on YOUR project. The five values, and the two that fail closed.
VALUES = [
    ('github_repository_id', 'gh api repos/OWNER/REPO --jq .id',
     'IMMUTABLE. A wrong value fails closed - nothing authenticates.'),
    ('github_repository',    'owner/repo',
     'MUTABLE. Readable in logs; never the only pin.'),
    ('the branch',           "DEPLOY_REF=refs/heads/main -> wif.tf's var.deploy_ref",
     'without it, anyone who can push a branch can deploy.'),
    ('project number',       'gcloud projects describe $P --format="value(projectNumber)"',
     'for the WIF pool resource name AND the IAP audience.'),
    ('the CI service account', 'sa-documind-cicd@$P.iam.gserviceaccount.com',
     'actAs is granted per account - list them, do not use project scope.'),
]
for what, how, why in VALUES:
    print(f'  {what:24} {how}')
    print(f'  {"":24} {why}')
print()
print('  Rows 1 and 3 are the two that tutorials leave out, and they fail in')
print('  opposite directions. A wrong repository_id fails CLOSED: nothing works,')
print('  you notice in ten minutes. A missing branch pin fails OPEN: everything')
print('  works, and every branch in your repository can deploy to production.')
print()
print('  You will not notice the second one. That is the whole argument for')
print('  pinning it on the day you create the pool, rather than after a review.')

## Cell 2: One Release, All the Way Through

What each stage actually proves.


In [ ]:
# One release, all the way through. What each stage proves.
STAGES = [
    ('eval gate (offline)',  'no credentials',              'the golden set is still sound'),
    ('build',                'first credentialed step',     'the image exists, tagged with the sha'),
    ('candidate revision',   'make release-candidate',      'it starts, on its own URL, taking no traffic'),
    ('eval gate (live)',     'against the candidate',       'it still answers, cites and REFUSES'),
    ('approval',             'a person',                    'now is a good time'),
    ('promote',              'make promote: a traffic flip', 'GET /version names the new sha'),
    ('if it is wrong',       'make rollback: the flip back', 'the previous revision never went away'),
]
print(f'  {"stage":22} {"runs":30} proves')
for s, runs, proves in STAGES:
    print(f'  {s:22} {runs:30} {proves}')
print()
print('  Component 8 needs ONE of these all the way through, plus a rollback.')
print('  Not a screenshot of a green workflow: the rollback, with /version output')
print('  from before and after, because that is the pair a stranger can check.')
print()
print('  And note stage 4. It is the first stage that needs the gate\'s two')
print('  identities - the roster member and the outsider - minted by the deploy')
print('  account, which is why 12.7 spent a lesson on a runner that can act as')
print('  them without storing a key.')
print()
print('  The Cloud Deploy path runs the same three gates with a second target:')
print('  staging, an approval, a 10% canary (clouddeploy.tf; path=clouddeploy, the')
print('  release-clouddeploy job). The candidate path is the one your fork ships')
print('  through first.')

## Cell 3: An SLO You Can Actually Hold

Alert on how fast you are spending the budget, not on a number being crossed.


In [ ]:
# An SLO you can actually hold, and the alert that fires before you break it.
#
# A burn-rate alert, not a threshold alert. The difference matters at 3am.
OBJECTIVE = 0.99                 # 99% of queries answered in under 4 seconds
WINDOW_DAYS = 30
BUDGET = 1 - OBJECTIVE           # 1% of requests may be slow


def burn(observed_bad_rate: float) -> float:
    """How many times faster than sustainable are we spending the budget?"""
    return observed_bad_rate / BUDGET


SCENARIOS = [
    ('a normal day',        0.004),
    ('a bad afternoon',     0.02),
    ('the context cache expired', 0.35),
]
print(f'  objective {OBJECTIVE:.0%} under 4s, {WINDOW_DAYS}-day window, '
      f'error budget {BUDGET:.0%}')
print()
print(f'  {"situation":22} {"bad rate":>9} {"burn":>7}  budget gone in')
for label, rate in SCENARIOS:
    b = burn(rate)
    days = WINDOW_DAYS / b if b else float('inf')
    when = f'{days:.1f} days' if days < WINDOW_DAYS else 'not this window'
    print(f'  {label:22} {rate:>8.1%} {b:>6.1f}x  {when}')
print()
print('  Page on a burn rate above 14x - that is the whole month\'s budget in')
print('  about two days. Do NOT page on "p95 > 4s": that fires on a slow minute')
print('  at 3am and teaches everybody to ignore it.')
print()
print('  The third row is the chaos script from 13.3, which is the point: the')
print('  same event you deliberately caused is the one the alert has to catch.')
print('  An alert nobody has ever seen fire is a hypothesis.')

## Cell 4: The Runbook Test

Not &ldquo;is it correct&rdquo;. Could a colleague follow it at 3am, while worried?


In [ ]:
# RUNBOOK.md - five entries, and the test is whether a colleague can run it.
ENTRIES = [
    ('roll back a bad deploy',
     'make rollback: traffic back to the revision before, which never went away (12.7)',
     'confirm with GET /version before and after; about fifteen seconds'),
    ('refresh a stale context cache',
     'client.caches.delete(name=...) then let the next query rebuild it',
     'symptom: answers cite a document that was replaced (10.2)'),
    ('replay the ingest DLQ',
     'make dlq to look; gcloud pubsub subscriptions pull ingest-dlq-sub --auto-ack to drain, then re-upload',
     'symptom: new uploads never appear (12.5)'),
    ('swap the generator model',
     'make candidate GENERATOR_MODEL=...; make eval-live API=<the candidate> BEFORE make promote',
     'the eval gate is the check, not your judgement (12.7)'),
    ('scale everything to zero',
     'make off (12.8); the nightly job does the same at 23:00 whether anyone remembers',
     'four things keep billing after the obvious step'),
]
for what, how, note in ENTRIES:
    print(f'  {what}')
    print(f'      {how}')
    print(f'      {note}')
    print()
print('  The test for a runbook entry is not "is it correct". It is: could')
print('  somebody who has never seen this system follow it at 3am, while worried?')
print()
print('  Which means every entry needs the SYMPTOM, not just the fix. "Replay')
print('  the DLQ" is useless to somebody who does not know that "new uploads')
print('  never appear" is what that looks like from outside.')

---

The rubric and the eight components are published in `course-bibles/capstone-rubric.md`. Score yourself against them before somebody else does.
